# Comparacion nota final y retiro: XGBoost vs CatBoost
Analiza el archivo configurado y compara que modelo predice mejor la nota final,
filtrando solo filas con `Retiro_Asignatura_Cat == 0`, y luego evalua la prediccion de retiro.


In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

# Configura rutas/columnas
#input_csv = r'Resultados_Modelo_Excel_MegaPipeline\resultados_asig_prereq_mp_corr_cat2026-01-28110609.csv'
#input_csv= r"Resultados_Modelo_v2\resultados_asig_xg_cat2026-01-29120929.csv"
#input_csv= r"Resultados_Modelo_Excel_MegaPipeline\union_febrero_resultados.csv"
input_csv = r"Resultados_Modelo_v2/resultados_asig_xg_cat2026-02-23171813.csv"
print(input_csv)
sep = ';'

col_real_retiro = 'Retiro_Asignatura_Cat'
col_nota_real = 'resultado_final'
col_nota_pred_cat = 'Prediccion_CAT'
col_nota_pred_xgb = 'Prediccion_XGB'
col_nota_pred_rf = ''

col_retiro_pred_xgb = 'Clasificacion_XGB'
col_retiro_pred_cat = 'Clasificacion_CAT'
labels_retiro = [0, 1]  # Cambia esta lista si las clases cambian.

def to_float(s):
    """Convierte notas con separador decimal ',' o '.' a float; no parseable -> NaN."""
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors='coerce')
    s_str = s.astype(str).str.strip()
    has_comma = s_str.str.contains(',', regex=False)
    has_dot = s_str.str.contains('.', regex=False)
    s_norm = s_str.copy()
    both = has_comma & has_dot
    s_norm = s_norm.where(~both, s_norm.str.replace('.', '', regex=False).str.replace(',', '.', regex=False))
    only_comma = has_comma & ~has_dot
    s_norm = s_norm.where(~only_comma, s_norm.str.replace(',', '.', regex=False))
    s_norm = s_norm.str.replace(' ', '', regex=False)
    return pd.to_numeric(s_norm, errors='coerce')

def normalize_class_series(s):
    """Normaliza etiquetas numericas o de texto para compararlas de forma consistente."""
    if pd.api.types.is_numeric_dtype(s):
        s_num = pd.to_numeric(s, errors='coerce')
    else:
        s_str = s.astype(str).str.strip()
        s_str = s_str.replace({'': np.nan, 'nan': np.nan, 'None': np.nan, '<NA>': np.nan})
        s_num = pd.to_numeric(s_str, errors='coerce')
        if s_num.notna().sum() != s_str.notna().sum():
            return s_str
    non_null = s_num.dropna()
    if len(non_null) == 0:
        return s_num.astype('Int64')
    if np.allclose(non_null, np.round(non_null)):
        return s_num.round().astype('Int64')
    return s_num

def compute_metrics(y, yhat):
    err = yhat - y
    mae = float(np.mean(np.abs(err)))
    rmse = float(np.sqrt(np.mean(err**2)))
    medae = float(np.median(np.abs(err)))
    return mae, rmse, medae

def evaluate_classification(y_true, y_pred, labels=None):
    valid = y_true.notna() & y_pred.notna()
    y_true_valid = y_true[valid].reset_index(drop=True)
    y_pred_valid = y_pred[valid].reset_index(drop=True)

    if len(y_true_valid) == 0:
        return None, None, np.nan, 0

    observed = [label for label in pd.unique(pd.concat([y_true_valid, y_pred_valid], ignore_index=True)) if pd.notna(label)]
    labels_eval = list(labels) if labels is not None else []
    for label in observed:
        if label not in labels_eval:
            labels_eval.append(label)

    if not labels_eval:
        labels_eval = observed

    confusion_df = pd.crosstab(
        pd.Categorical(y_true_valid, categories=labels_eval),
        pd.Categorical(y_pred_valid, categories=labels_eval),
        rownames=['Real'],
        colnames=['Prediccion'],
        dropna=False,
    )

    metrics_rows = []
    for label in labels_eval:
        tp = confusion_df.loc[label, label]
        predicted_total = confusion_df[label].sum()
        real_total = confusion_df.loc[label].sum()
        precision = tp / predicted_total if predicted_total else np.nan
        recall = tp / real_total if real_total else np.nan
        metrics_rows.append({'categoria': label, 'precision': precision, 'recall': recall})

    metrics_df = pd.DataFrame(metrics_rows).set_index('categoria')
    total = int(confusion_df.to_numpy().sum())
    accuracy = float(np.trace(confusion_df.to_numpy()) / total) if total else np.nan
    return confusion_df, metrics_df, accuracy, int(valid.sum())

df = pd.read_csv(input_csv, sep=sep, low_memory=False)

real_ret = normalize_class_series(df[col_real_retiro])
nota_real = to_float(df[col_nota_real])
nota_cat = to_float(df[col_nota_pred_cat])
nota_xgb = to_float(df[col_nota_pred_xgb])

mask = real_ret == 0
mask_base = mask & nota_real.notna()

# Metricas por modelo (solo donde exista prediccion)
m_cat = mask_base & nota_cat.notna()
m_xgb = mask_base & nota_xgb.notna()

y_cat = nota_real[m_cat]
yhat_cat = nota_cat[m_cat]
y_xgb = nota_real[m_xgb]
yhat_xgb = nota_xgb[m_xgb]

print('=== Comparacion de nota final ===')
print('Filas totales:', len(df))
print('Filas retiro==0:', int(mask.sum()))
print('Filas con nota real:', int(mask_base.sum()))

if len(y_cat) > 0:
    mae, rmse, medae = compute_metrics(y_cat, yhat_cat)
    print(f'CatBoost -> MAE: {mae:.4f}, RMSE: {rmse:.4f}, MedAE: {medae:.4f} (n={len(y_cat)})')
else:
    print('CatBoost -> sin filas validas')

if len(y_xgb) > 0:
    mae, rmse, medae = compute_metrics(y_xgb, yhat_xgb)
    print(f'XGBoost -> MAE: {mae:.4f}, RMSE: {rmse:.4f}, MedAE: {medae:.4f} (n={len(y_xgb)})')
else:
    print('XGBoost -> sin filas validas')

# Comparacion directa (ambos disponibles)
mask_both = mask_base & nota_cat.notna() & nota_xgb.notna()
if mask_both.any():
    y = nota_real[mask_both]
    cat = nota_cat[mask_both]
    xgb = nota_xgb[mask_both]
    err_cat = np.abs(cat - y)
    err_xgb = np.abs(xgb - y)
    win_cat = int((err_cat < err_xgb).sum())
    win_xgb = int((err_xgb < err_cat).sum())
    tie = int((err_cat == err_xgb).sum())
    print('Comparacion fila a fila (ambos disponibles):')
    print('  Gana CatBoost:', win_cat)
    print('  Gana XGBoost:', win_xgb)
    print('  Empates:', tie)
else:
    print('No hay filas con ambas predicciones disponibles')

print()
print('=== Comparacion de prediccion de retiro ===')
retiro_real = normalize_class_series(df[col_real_retiro])
retiro_cat = normalize_class_series(df[col_retiro_pred_cat])
retiro_xgb = normalize_class_series(df[col_retiro_pred_xgb])

for model_name, retiro_pred in [('CatBoost', retiro_cat), ('XGBoost', retiro_xgb)]:
    confusion_df, metrics_df, accuracy, n_valid = evaluate_classification(retiro_real, retiro_pred, labels_retiro)
    print()
    print(f'{model_name} - filas validas para retiro: {n_valid}')

    if n_valid == 0:
        print(f'{model_name} -> sin filas validas para evaluar retiro')
        continue

    print('Matriz de confusion:')
    display(confusion_df)

    print('Precision y recall por categoria:')
    display(metrics_df.round(4))

    print(f'Accuracy general: {accuracy:.4f}')


Resultados_Modelo_v2/resultados_asig_xg_cat2026-02-23171813.csv
=== Comparacion de nota final ===
Filas totales: 21607
Filas retiro==0: 20450
Filas con nota real: 20450
CatBoost -> MAE: 0.4700, RMSE: 0.6349, MedAE: 0.3700 (n=20450)
XGBoost -> MAE: 0.4567, RMSE: 0.6266, MedAE: 0.3500 (n=20450)
Comparacion fila a fila (ambos disponibles):
  Gana CatBoost: 9796
  Gana XGBoost: 10339
  Empates: 315

=== Comparacion de prediccion de retiro ===

CatBoost - filas validas para retiro: 21607
Matriz de confusion:


Prediccion,0,1
Real,,
0,19709,741
1,857,300


Precision y recall por categoria:


,precision,recall
categoria,,
0,0.9583,0.9638
1,0.2882,0.2593


Accuracy general: 0.9260

XGBoost - filas validas para retiro: 21607
Matriz de confusion:


Prediccion,0,1
Real,,
0,19882,568
1,964,193


Precision y recall por categoria:


,precision,recall
categoria,,
0,0.9538,0.9722
1,0.2536,0.1668


Accuracy general: 0.9291
